In [1]:
import sys
from pathlib import Path

package_root = str(Path.cwd().parent)
if package_root not in sys.path:
    sys.path.append(package_root)

In [2]:
from __future__ import annotations
import pandas as pd 
from pathlib import Path
from collections.abc import Iterable
import json

In [3]:
DATASET_FOLDER = "../../../dataset/processed-data"
DEVICE_MAPPING_FOLDER = DATASET_FOLDER + "/device-mapping"
INPUT_FOLDER = DATASET_FOLDER + "/input-data"
DATASET1 = "dataset1"
DATASET2 = "dataset2"

In [4]:
MAPPING_LIST_1 = f"{DEVICE_MAPPING_FOLDER}/{DATASET1}/list_of_devices_processed.csv"
MAPPING_LIST_2 = f"{DEVICE_MAPPING_FOLDER}/{DATASET2}/list_of_devices_processed.csv"
INPUT_1 = f"{INPUT_FOLDER}/{DATASET1}"
INPUT_2 = f"{INPUT_FOLDER}/{DATASET2}"

In [5]:
device_mapping_1 = pd.read_csv(MAPPING_LIST_1)
device_dict_1 = (
    device_mapping_1
    .set_index("mac_address")
    .drop(columns=["mac_address"], errors="ignore")
    .to_dict(orient="index")
)
device_dict_1

{'d0:52:a8:00:67:5e': {'device_name': 'Smart Things',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'hub',
  'vendor': 'Samsung SmartThings'},
 '44:65:0d:56:cc:d3': {'device_name': 'Amazon Echo',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'smart speaker',
  'vendor': 'Amazon'},
 '70:ee:50:18:34:43': {'device_name': 'Netatmo Welcome',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Netatmo'},
 'f4:f2:6d:93:51:f1': {'device_name': 'TP-Link Day Night Cloud camera',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'TP-Link'},
 '00:16:6c:ab:6b:88': {'device_name': 'Samsung SmartCam',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Samsung'},
 '30:8c:fb:2f:e4:b2': {'device_name': 'Dropcam',
  'ip_address': nan,
  'is_iot_device': 1,
  'device_type': 'camera',
  'vendor': 'Dropcam'},
 '00:62:6e:51:27:2e': {'device_name': 'Insteon Camera Wired',
  'ip_address': 

In [6]:
csv_files_1 = [f"{INPUT_1}/{f.name}" for f in sorted(Path(INPUT_1).iterdir()) if f.is_file() and f.suffix == ".csv"]
for file in csv_files_1:
    print(file)

../../../dataset/processed-data/input-data/dataset1/16-09-23.csv
../../../dataset/processed-data/input-data/dataset1/16-09-24.csv
../../../dataset/processed-data/input-data/dataset1/16-09-25.csv
../../../dataset/processed-data/input-data/dataset1/16-09-26.csv
../../../dataset/processed-data/input-data/dataset1/16-09-27.csv
../../../dataset/processed-data/input-data/dataset1/16-09-28.csv
../../../dataset/processed-data/input-data/dataset1/16-09-29.csv
../../../dataset/processed-data/input-data/dataset1/16-09-30.csv
../../../dataset/processed-data/input-data/dataset1/16-10-01.csv
../../../dataset/processed-data/input-data/dataset1/16-10-02.csv
../../../dataset/processed-data/input-data/dataset1/16-10-03.csv
../../../dataset/processed-data/input-data/dataset1/16-10-04.csv
../../../dataset/processed-data/input-data/dataset1/16-10-05.csv
../../../dataset/processed-data/input-data/dataset1/16-10-06.csv
../../../dataset/processed-data/input-data/dataset1/16-10-07.csv
../../../dataset/processe

In [7]:
# Outgoing volume
def extract_data_from_pcap(
    file_paths: Iterable[str]
) -> pd.DataFrame:
    columns = [
        "frame.time_epoch",
        "frame.len",
        "eth.src",
        "eth.dst",
    ]

    all_data = []

    for file_path in file_paths:
        raw_df = pd.read_csv(
            file_path,
            usecols=columns,
            dtype={
                "frame.time_epoch": "float64",
                "frame.len": "int32",
                "eth.src": "category",
                "eth.dst": "category",
            }
        )

        raw_df["capture_id"] = Path(file_path).stem
        all_data.append(raw_df)


    res = pd.concat(all_data, ignore_index=True)
    res = res.sort_values(
        ["frame.time_epoch", "eth.src"],
        ignore_index=True,
    )

    return res


In [8]:
all_data = extract_data_from_pcap(csv_files_1)

In [9]:
Path("dataset1").mkdir(parents=True, exist_ok=True)
all_data.to_parquet(
    "dataset1/all_traffic.parquet",
    index=False
)

In [10]:
eda_df = all_data.copy()

In [11]:
capture_counts = (
    eda_df.groupby("eth.src")["capture_id"]
      .nunique()
)

single_capture_devices = capture_counts[
    capture_counts == 1
].index

single_capture_info = (
    eda_df[eda_df["eth.src"].isin(single_capture_devices)]
    [
        ["eth.src"]
    ]
    .drop_duplicates()
    .sort_values("eth.src")
)
# single_capture_info = label_features(single_capture_info, device_dict_1)
# single_capture_info_with_label = label_features(single_capture_info, device_dict_1)
# single_capture_info
eth_src_list = single_capture_info["eth.src"].dropna().unique().tolist()
print(f"Number of devices seen in only one capture: {len(eth_src_list)}")
for mac in eth_src_list:
    if mac in device_dict_1:
        print(f"mac '{mac}': {device_dict_1[mac]['device_name']}")
    else:
        print(f"mac '{mac}': not found in device list")

Number of devices seen in only one capture: 6
mac '00:24:e4:10:ee:4c': not found in device list
mac '30:8c:fb:b6:ea:45': Nest Dropcam
mac '8a:05:81:fa:cc:14': not found in device list
mac 'd2:13:91:23:2a:58': not found in device list
mac 'e8:ab:fa:19:de:4f': Insteon Camera Wireless
mac 'f4:5c:89:93:cc:85': MacBook/Iphone


In [12]:
single_capture_set = set(eth_src_list)
device_to_be_include = {mac: device_dict_1[mac] for mac in device_dict_1 if mac not in single_capture_set}

In [13]:
list_of_devices_to_be_classified = list(device_to_be_include.keys())
print(f"Total: {len(list_of_devices_to_be_classified)} devices")
for mac in list_of_devices_to_be_classified:
    print(f"mac '{mac}': {device_to_be_include[mac]['device_name']},\tis_iot: {device_to_be_include[mac]['is_iot_device']}, \tdevice_type: {device_to_be_include[mac]['device_type']}")

Total: 28 devices
mac 'd0:52:a8:00:67:5e': Smart Things,	is_iot: 1, 	device_type: hub
mac '44:65:0d:56:cc:d3': Amazon Echo,	is_iot: 1, 	device_type: smart speaker
mac '70:ee:50:18:34:43': Netatmo Welcome,	is_iot: 1, 	device_type: camera
mac 'f4:f2:6d:93:51:f1': TP-Link Day Night Cloud camera,	is_iot: 1, 	device_type: camera
mac '00:16:6c:ab:6b:88': Samsung SmartCam,	is_iot: 1, 	device_type: camera
mac '30:8c:fb:2f:e4:b2': Dropcam,	is_iot: 1, 	device_type: camera
mac '00:62:6e:51:27:2e': Insteon Camera Wired,	is_iot: 1, 	device_type: camera
mac '00:24:e4:11:18:a8': Withings Smart Baby Monitor,	is_iot: 1, 	device_type: baby monitor
mac 'ec:1a:59:79:f4:89': Belkin Wemo switch,	is_iot: 1, 	device_type: smart switch
mac '50:c7:bf:00:56:39': TP-Link Smart plug,	is_iot: 1, 	device_type: smart plug
mac '74:c6:3b:29:d7:1d': iHome,	is_iot: 1, 	device_type: smart plug
mac 'ec:1a:59:83:28:11': Belkin wemo motion sensor,	is_iot: 1, 	device_type: motion sensor
mac '18:b4:30:25:be:e4': NEST Protect s

In [15]:
with open("dataset1/device_to_be_include.json", "w") as f:
    json.dump(device_to_be_include, f, indent=4)